In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  SUPERFERMION × QISKIT — IBM QPU SHOWCASE                                  ║
║  Complex circuits | Cross-framework validation | Live IBM QPU execution    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, time, math, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Notebook is in notebooks/ folder — root is parent
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Load .env with IBM token
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

IBM_TOKEN = os.getenv('IBM_QUANTUM_TOKEN', '')
print(f"IBM_TOKEN loaded: {'✓' if IBM_TOKEN else '✗ NOT FOUND — check .env'}")

import numpy as np
np.set_printoptions(precision=8, suppress=True, linewidth=120)

import superfermion as sf
from superfermion.backends.registry import BackendRegistry
from superfermion.backends.stabilizer import StabilizerBackend
from superfermion.backends.mps import MPSSimulatorBackend
from superfermion.observables.core import SparsePauliOp

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector as QkStatevector, SparsePauliOp as QkPauliOp
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

print(f"\n  SF version: {sf.__version__ if hasattr(sf, '__version__') else 'dev'}")
print(f"  NumPy: {np.__version__}")
print("  All imports OK ✓")

RESULTS = {}  # accumulates all run results


## 1. Probe All SF Simulation Backends


In [ ]:
print("=" * 70)
print("  BACKEND PROBE — Which backends work?")
print("=" * 70)

ALL_BACKENDS = [
    "statevector", "rust", "mps", "jax", "jax_mps",
    "stabilizer", "density_matrix", "singularity",
    "cuda", "cuda_mps",
]

probe = sf.Circuit(2); probe.h(0); probe.cx(0, 1)
working = []
failed = []

for name in ALL_BACKENDS:
    try:
        be = BackendRegistry.get_backend(name)
        r = sf.run(probe, backend=name, shots=128)
        working.append(name)
        sv_ok = r.statevector is not None
        ct_ok = r.counts is not None
        print(f"  {name:<18s} ✓  (sv={sv_ok}, counts={ct_ok})")
    except Exception as e:
        failed.append((name, str(e)[:60]))
        print(f"  {name:<18s} ✗  {str(e)[:60]}")

print(f"\n  Working: {len(working)}/{len(ALL_BACKENDS)} — {working}")
RESULTS['working_backends'] = working


## 2. Build Complex Circuits (SF + Qiskit equivalents)


In [ ]:
print("=" * 70)
print("  COMPLEX CIRCUIT BUILDERS")
print("=" * 70)

# --- Bell State ---
def make_bell_sf():
    c = sf.Circuit(2); c.h(0); c.cx(0, 1)
    return c

def make_bell_qk():
    qc = QuantumCircuit(2); qc.h(0); qc.cx(0, 1)
    return qc

# --- GHZ State (n-qubit) ---
def make_ghz_sf(n=3):
    c = sf.Circuit(n); c.h(0)
    for i in range(n - 1): c.cx(i, i + 1)
    return c

def make_ghz_qk(n=3):
    qc = QuantumCircuit(n); qc.h(0)
    for i in range(n - 1): qc.cx(i, i + 1)
    return qc

# --- QFT (n-qubit) ---
def make_qft_sf(n=4):
    c = sf.Circuit(n)
    for j in range(n):
        c.h(j)
        for k in range(j + 1, n):
            c.cp(math.pi / (2 ** (k - j)), k, j)
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c

def make_qft_qk(n=4):
    qc = QuantumCircuit(n)
    for j in range(n):
        qc.h(j)
        for k in range(j + 1, n):
            qc.cp(math.pi / (2 ** (k - j)), k, j)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

# --- QAOA-p2 MaxCut ---
def make_qaoa_sf(n=4):
    c = sf.Circuit(n)
    gamma = [0.3, 0.5]; beta = [0.2, 0.4]
    for q in range(n): c.h(q)
    for p in range(2):
        for i in range(n - 1):
            c.cx(i, i + 1); c.rz(2 * gamma[p], i + 1); c.cx(i, i + 1)
        for q in range(n): c.rx(2 * beta[p], q)
    return c

def make_qaoa_qk(n=4):
    qc = QuantumCircuit(n)
    gamma = [0.3, 0.5]; beta = [0.2, 0.4]
    for q in range(n): qc.h(q)
    for p in range(2):
        for i in range(n - 1):
            qc.cx(i, i + 1); qc.rz(2 * gamma[p], i + 1); qc.cx(i, i + 1)
        for q in range(n): qc.rx(2 * beta[p], q)
    return qc

# --- Heisenberg Trotter ---
def make_heisenberg_sf(n=4, steps=8):
    J, dt = 1.0, 0.05
    c = sf.Circuit(n)
    for _ in range(steps):
        for i in range(n - 1):
            # RXX
            c.h(i); c.h(i + 1)
            c.cx(i, i + 1); c.rz(2 * J * dt, i + 1); c.cx(i, i + 1)
            c.h(i); c.h(i + 1)
            # RYY
            c.rx(math.pi / 2, i); c.rx(math.pi / 2, i + 1)
            c.cx(i, i + 1); c.rz(2 * J * dt, i + 1); c.cx(i, i + 1)
            c.rx(-math.pi / 2, i); c.rx(-math.pi / 2, i + 1)
            # RZZ
            c.cx(i, i + 1); c.rz(2 * J * dt, i + 1); c.cx(i, i + 1)
    return c

def make_heisenberg_qk(n=4, steps=8):
    J, dt = 1.0, 0.05
    qc = QuantumCircuit(n)
    for _ in range(steps):
        for i in range(n - 1):
            qc.h(i); qc.h(i + 1)
            qc.cx(i, i + 1); qc.rz(2 * J * dt, i + 1); qc.cx(i, i + 1)
            qc.h(i); qc.h(i + 1)
            qc.rx(math.pi / 2, i); qc.rx(math.pi / 2, i + 1)
            qc.cx(i, i + 1); qc.rz(2 * J * dt, i + 1); qc.cx(i, i + 1)
            qc.rx(-math.pi / 2, i); qc.rx(-math.pi / 2, i + 1)
            qc.cx(i, i + 1); qc.rz(2 * J * dt, i + 1); qc.cx(i, i + 1)
    return qc

# --- Random Clifford ---
def make_clifford_sf(n=6, layers=10, seed=42):
    rng = np.random.default_rng(seed)
    c = sf.Circuit(n)
    for _ in range(layers):
        for q in range(n):
            g = int(rng.integers(0, 3))
            if g == 0: c.h(q)
            elif g == 1: c.s(q)
        for i in range(0, n - 1, 2): c.cx(i, i + 1)
        for i in range(1, n - 1, 2): c.cx(i, i + 1)
    return c

def make_clifford_qk(n=6, layers=10, seed=42):
    rng = np.random.default_rng(seed)
    qc = QuantumCircuit(n)
    for _ in range(layers):
        for q in range(n):
            g = int(rng.integers(0, 3))
            if g == 0: qc.h(q)
            elif g == 1: qc.s(q)
        for i in range(0, n - 1, 2): qc.cx(i, i + 1)
        for i in range(1, n - 1, 2): qc.cx(i, i + 1)
    return qc

# --- Grover 2-qubit ---
def make_grover_sf(target='11'):
    c = sf.Circuit(2)
    c.h(0); c.h(1)  # superposition
    # Oracle for |11>
    if target == '11':
        c.cz(0, 1)
    elif target == '10':
        c.x(1); c.cz(0, 1); c.x(1)
    elif target == '01':
        c.x(0); c.cz(0, 1); c.x(0)
    elif target == '00':
        c.x(0); c.x(1); c.cz(0, 1); c.x(0); c.x(1)
    # Diffusion
    c.h(0); c.h(1); c.x(0); c.x(1)
    c.cz(0, 1)
    c.x(0); c.x(1); c.h(0); c.h(1)
    return c

print("  Circuit builders ready:")
print("    Bell (2q), GHZ (n), QFT (n), QAOA-p2 (n),")
print("    Heisenberg Trotter (n), Random Clifford (n), Grover (2q)")

# Quick sanity: build & print gate counts
for name, fn, n in [("Bell", make_bell_sf, 2), ("GHZ-5", make_ghz_sf, 5),
                     ("QFT-6", make_qft_sf, 6), ("QAOA-4", make_qaoa_sf, 4),
                     ("Heis-4", make_heisenberg_sf, 4), ("Cliff-6", make_clifford_sf, 6)]:
    c = fn() if n is None else fn(n)
    print(f"    {name:12s}: {c.n_qubits}q, {c.gate_count} gates")


## 3. SF Multi-Backend Simulation — Performance Comparison


In [ ]:
print("=" * 70)
print("  SF MULTI-BACKEND SIMULATION (n=4..10)")
print("=" * 70)

import gc, psutil

def track(fn):
    gc.collect()
    proc = psutil.Process(os.getpid())
    rss0 = proc.memory_info().rss
    t0 = time.perf_counter()
    result = fn()
    dt = (time.perf_counter() - t0) * 1000.0
    rss1 = proc.memory_info().rss
    mem_mb = (rss1 - rss0) / 1024 / 1024
    return result, dt, mem_mb

SIM_BACKENDS = [b for b in ["statevector", "rust", "jax", "mps", "stabilizer"] if b in working]
CIRCUITS = {
    "GHZ-4": make_ghz_sf(4), "GHZ-8": make_ghz_sf(8),
    "QFT-5": make_qft_sf(5), "QFT-8": make_qft_sf(8),
    "QAOA-4": make_qaoa_sf(4), "QAOA-8": make_qaoa_sf(8),
    "Cliff-6": make_clifford_sf(6), "Cliff-10": make_clifford_sf(10),
    "Heis-4": make_heisenberg_sf(4),
}

print(f"\n  {'Circuit':<12s} {'Backend':<18s} {'Time(ms)':<12s} {'Mem(MB)':<10s} {'Fidelity':<14s}")
print("  " + "-" * 70)

all_sim_results = []

for cname, circuit in CIRCUITS.items():
    n = circuit.n_qubits
    # Get reference statevector for fidelity
    ref_sv = None
    if "statevector" in working:
        try:
            r = BackendRegistry.get_backend("statevector").run(circuit, shots=0)
            ref_sv = np.asarray(r.statevector, dtype=np.complex128)
        except:
            pass

    for bk in SIM_BACKENDS:
        try:
            be = BackendRegistry.get_backend(bk)
            sv, dt, mem = track(lambda: be.run(circuit, shots=0))
            sv_arr = np.asarray(sv.statevector, dtype=np.complex128) if sv.statevector is not None else None
            fid_str = ""
            if ref_sv is not None and sv_arr is not None and sv_arr.size == ref_sv.size:
                sv_lsb = sv_arr.reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1)
                ref_lsb = ref_sv.reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1)
                fid = float(abs(np.vdot(sv_lsb, ref_lsb)))
                fid_str = f"{fid:.10f}"
            print(f"  {cname:<12s} {bk:<18s} {dt:<12.2f} {mem:<+10.2f} {fid_str:<14s}")
            all_sim_results.append({"circuit": cname, "backend": bk, "time_ms": dt, "mem_mb": mem, "fidelity": fid_str})
        except Exception as e:
            print(f"  {cname:<12s} {bk:<18s} FAIL — {str(e)[:40]}")

RESULTS['sim_results'] = all_sim_results
print(f"\n  Done — {len(all_sim_results)} simulation runs")


## 4. Cross-Framework Validation — SF vs Qiskit Aer


In [ ]:
print("=" * 70)
print("  CROSS-FRAMEWORK VALIDATION — SF vs Qiskit Aer")
print("=" * 70)

cross_tests = [
    ("Bell-2", make_bell_sf(), make_bell_qk()),
    ("GHZ-4", make_ghz_sf(4), make_ghz_qk(4)),
    ("GHZ-8", make_ghz_sf(8), make_ghz_qk(8)),
    ("QFT-4", make_qft_sf(4), make_qft_qk(4)),
    ("QFT-6", make_qft_sf(6), make_qft_qk(6)),
    ("QAOA-4", make_qaoa_sf(4), make_qaoa_qk(4)),
    ("QAOA-6", make_qaoa_sf(6), make_qaoa_qk(6)),
    ("Cliff-6", make_clifford_sf(6), make_clifford_qk(6)),
    ("Heis-4", make_heisenberg_sf(4), make_heisenberg_qk(4)),
]

def sf_to_lsb(sv, n):
    return np.asarray(sv).reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1)

aer_sim = AerSimulator(method="statevector")

print(f"\n  {'Circuit':<12s} {'SF <Z0Z1>':<18s} {'Aer <Z0Z1>':<18s} {'Diff':<12s} {'Fidelity':<14s} {'Verdict':<10s}")
print("  " + "-" * 88)

cross_results = []
for name, c_sf, c_qk in cross_tests:
    n = c_sf.n_qubits
    try:
        # SF statevector
        r_sf = BackendRegistry.get_backend("statevector").run(c_sf, shots=0)
        sv_sf = np.asarray(r_sf.statevector, dtype=np.complex128)
        sv_sf_lsb = sf_to_lsb(sv_sf, n)

        # Aer statevector
        qc2 = c_qk.copy(); qc2.save_statevector()
        sv_aer = np.asarray(aer_sim.run(qc2).result().get_statevector(), dtype=np.complex128)

        # Fidelity
        fid = float(abs(np.vdot(sv_sf_lsb, sv_aer)))

        # <Z0Z1> expectation
        idx = np.arange(1 << n)
        parity_sf = ((idx >> (n - 1)) & 1) ^ ((idx >> (n - 2)) & 1)
        z_sf = float(np.sum(np.where(parity_sf == 0, 1.0, -1.0) * np.abs(sv_sf) ** 2))

        parity_aer = (idx & 1) ^ ((idx >> 1) & 1)
        z_aer = float(np.sum(np.where(parity_aer == 0, 1.0, -1.0) * np.abs(sv_aer) ** 2))

        z_diff = abs(z_sf - z_aer)
        verdict = "PASS" if fid > 1 - 1e-12 else "FAIL"
        print(f"  {name:<12s} {z_sf:<+18.12f} {z_aer:<+18.12f} {z_diff:<12.2e} {fid:<14.12f} {verdict:<10s}")
        cross_results.append({"name": name, "z_sf": z_sf, "z_aer": z_aer, "diff": z_diff, "fidelity": fid, "pass": fid > 1 - 1e-12})
    except Exception as e:
        print(f"  {name:<12s} ERROR — {str(e)[:50]}")

RESULTS['cross_validation'] = cross_results
passed = sum(1 for r in cross_results if r['pass'])
print(f"\n  Cross-framework: {passed}/{len(cross_results)} passed")


## 5. IBM QPU — Connect, List Backends, Noise Data


In [ ]:
print("=" * 70)
print("  IBM QUANTUM — Connect & Explore")
print("=" * 70)

if not IBM_TOKEN:
    print("  No IBM token — skipping QPU section")
else:
    print("  Connecting to IBM Quantum...")
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
    print("  Connected!")

    # List backends
    all_backends = service.backends()
    print(f"\n  Available backends: {len(all_backends)}")
    print(f"  {'Name':<24s} {'Qubits':<8s} {'Pending':<10s} {'Status':<12s}")
    print("  " + "-" * 56)

    ibm_backends_info = []
    for b in all_backends:
        try:
            nq = b.num_qubits
            pending = getattr(b.status(), 'pending_jobs', '?') if b.status() else '?'
            st = str(b.status().status) if b.status() else '?'
            print(f"  {b.name:<24s} {nq:<8d} {str(pending):<10s} {st:<12s}")
            ibm_backends_info.append({"name": b.name, "qubits": nq, "pending": pending, "status": st})
        except Exception as e:
            print(f"  {b.name:<24s} ? — {str(e)[:30]}")

    RESULTS['ibm_backends'] = ibm_backends_info

    # Pick best available backend
    if ibm_backends_info:
        best = max(ibm_backends_info, key=lambda x: (x['qubits'], -int(x['pending']) if isinstance(x['pending'], int) else 0))
        TARGET_BACKEND = best['name']
        print(f"\n  Target backend: {TARGET_BACKEND} ({best['qubits']}q)")
        RESULTS['target_backend'] = TARGET_BACKEND

    # Noise data
    print(f"\n  --- Noise Data: {TARGET_BACKEND} ---")
    try:
        ibmq_backend = service.backend(TARGET_BACKEND)
        props = ibmq_backend.properties()
        nq = ibmq_backend.num_qubits
        print(f"  Qubits: {nq}")
        print(f"  {'Qubit':<8s} {'T1(us)':<12s} {'T2(us)':<12s} {'ReadoutErr':<12s} {'GateErr(CX)':<14s}")
        print("  " + "-" * 60)
        for i in range(min(nq, 10)):
            t1 = props.t1(i) * 1e6 if props.t1(i) else 0
            t2 = props.t2(i) * 1e6 if props.t2(i) else 0
            ro = props.readout_error(i) if hasattr(props, 'readout_error') else 0
            try:
                cx_err = props.gate_error('cx', [i, (i+1) % nq])
            except:
                cx_err = 0
            print(f"  {i:<8d} {t1:<12.1f} {t2:<12.1f} {ro:<12.4f} {cx_err:<14.4f}")
    except Exception as e:
        print(f"  Could not fetch noise data: {e}")


## 6. IBM QPU — Submit Bell Circuit (Warm-Up)


In [ ]:
print("=" * 70)
print("  IBM QPU — Bell State Submission")
print("=" * 70)

if not IBM_TOKEN:
    print("  Skipped — no IBM token")
else:
    bell_sf = make_bell_sf()
    print(f"  SF Bell: {bell_sf.n_qubits}q, {bell_sf.gate_count} gates")

    from superfermion.bridge import to_qiskit
    bell_qk = to_qiskit(bell_sf)
    bell_qk.measure_all()
    print(f"  Qiskit circuit: {bell_qk.num_qubits}q, depth={bell_qk.depth()}")
    print(f"  Drawing:\n{bell_qk.draw('text', fold=60)}")

    print(f"\n  Transpiling for {TARGET_BACKEND}...")
    ibmq_backend = service.backend(TARGET_BACKEND)
    pm = generate_preset_pass_manager(optimization_level=3, backend=ibmq_backend)
    isa_bell = pm.run(bell_qk)
    print(f"  ISA circuit: depth={isa_bell.depth()}, ops={isa_bell.count_ops()}")

    print(f"\n  Submitting to IBM {TARGET_BACKEND}...")
    sampler = Sampler(mode=ibmq_backend)
    job_bell = sampler.run([isa_bell], shots=4096)
    print(f"  Job ID: {job_bell.job_id()}")
    print(f"  Status: {job_bell.status()}")

    RESULTS['ibm_bell_job_id'] = job_bell.job_id()

    print(f"\n  Waiting for result (timeout 300s)...")
    t0 = time.perf_counter()
    try:
        result_bell = job_bell.result(timeout=300)
        dt = time.perf_counter() - t0
        print(f"  Result received in {dt:.1f}s")

        pub = result_bell[0]
        if hasattr(pub.data, 'meas'):
            counts = pub.data.meas.get_counts()
        elif hasattr(pub.data, 'c'):
            counts = pub.data.c.get_counts()
        else:
            counts = pub.data[next(iter(pub.data._fields))].get_counts()

        total = sum(counts.values())
        print(f"\n  Counts ({total} shots):")
        for bs, cnt in sorted(counts.items(), key=lambda x: -x[1]):
            bar = chr(9608) * int(cnt / total * 50)
            print(f"    {bs}: {cnt:5d} ({cnt/total*100:5.1f}%) {bar}")

        bell_fid = (counts.get('00', 0) + counts.get('11', 0)) / total
        print(f"\n  Bell fidelity (|00>+|11>): {bell_fid*100:.1f}%")
        print(f"  Expected on real HW: ~90-99%")
        RESULTS['ibm_bell'] = {'counts': counts, 'fidelity': bell_fid, 'latency_s': dt}
    except Exception as e:
        dt = time.perf_counter() - t0
        print(f"  Timeout/Error after {dt:.1f}s: {e}")
        print(f"  Job ID {job_bell.job_id()} still running — use IBM dashboard")
        RESULTS['ibm_bell'] = {'job_id': job_bell.job_id(), 'status': str(job_bell.status()), 'error': str(e)[:120]}


## 7. IBM QPU — GHZ-3 State (Entanglement Witness)


In [ ]:
print("=" * 70)
print("  IBM QPU — GHZ-3 State")
print("=" * 70)

if not IBM_TOKEN:
    print("  Skipped — no IBM token")
else:
    ghz_sf = make_ghz_sf(3)
    print(f"  SF GHZ-3: {ghz_sf.gate_count} gates")

    from superfermion.bridge import to_qiskit
    ghz_qk = to_qiskit(ghz_sf)
    ghz_qk.measure_all()

    ibmq_backend = service.backend(TARGET_BACKEND)
    pm = generate_preset_pass_manager(optimization_level=3, backend=ibmq_backend)
    isa_ghz = pm.run(ghz_qk)
    print(f"  ISA GHZ-3: depth={isa_ghz.depth()}, ops={isa_ghz.count_ops()}")

    print(f"\n  Submitting to {TARGET_BACKEND}...")
    sampler = Sampler(mode=ibmq_backend)
    job_ghz = sampler.run([isa_ghz], shots=4096)
    print(f"  Job ID: {job_ghz.job_id()}")
    RESULTS['ibm_ghz3_job_id'] = job_ghz.job_id()

    t0 = time.perf_counter()
    try:
        result_ghz = job_ghz.result(timeout=300)
        dt = time.perf_counter() - t0
        print(f"  Result in {dt:.1f}s")

        pub = result_ghz[0]
        if hasattr(pub.data, 'meas'):
            counts = pub.data.meas.get_counts()
        elif hasattr(pub.data, 'c'):
            counts = pub.data.c.get_counts()
        else:
            counts = pub.data[next(iter(pub.data._fields))].get_counts()

        total = sum(counts.values())
        print(f"\n  Counts ({total} shots):")
        for bs, cnt in sorted(counts.items(), key=lambda x: -x[1]):
            bar = chr(9608) * int(cnt / total * 50)
            print(f"    {bs}: {cnt:5d} ({cnt/total*100:5.1f}%) {bar}")

        ghz_fid = (counts.get('000', 0) + counts.get('111', 0)) / total
        print(f"\n  GHZ fidelity (|000>+|111>): {ghz_fid*100:.1f}%")
        RESULTS['ibm_ghz3'] = {'counts': counts, 'fidelity': ghz_fid, 'latency_s': dt}
    except Exception as e:
        dt = time.perf_counter() - t0
        print(f"  Timeout/Error after {dt:.1f}s: {e}")
        RESULTS['ibm_ghz3'] = {'job_id': job_ghz.job_id(), 'status': str(job_ghz.status()), 'error': str(e)[:120]}


## 8. IBM QPU — Grover Search (2-qubit, oracle for |11>)


In [ ]:
print("=" * 70)
print("  IBM QPU — Grover Search |11>")
print("=" * 70)

if not IBM_TOKEN:
    print("  Skipped — no IBM token")
else:
    grover_sf = make_grover_sf('11')
    print(f"  SF Grover: {grover_sf.gate_count} gates")

    from superfermion.bridge import to_qiskit
    grover_qk = to_qiskit(grover_sf)
    grover_qk.measure_all()

    ibmq_backend = service.backend(TARGET_BACKEND)
    pm = generate_preset_pass_manager(optimization_level=3, backend=ibmq_backend)
    isa_grover = pm.run(grover_qk)
    print(f"  ISA Grover: depth={isa_grover.depth()}, ops={isa_grover.count_ops()}")

    print(f"\n  Submitting to {TARGET_BACKEND}...")
    sampler = Sampler(mode=ibmq_backend)
    job_grover = sampler.run([isa_grover], shots=4096)
    print(f"  Job ID: {job_grover.job_id()}")
    RESULTS['ibm_grover_job_id'] = job_grover.job_id()

    t0 = time.perf_counter()
    try:
        result_grover = job_grover.result(timeout=300)
        dt = time.perf_counter() - t0
        print(f"  Result in {dt:.1f}s")

        pub = result_grover[0]
        if hasattr(pub.data, 'meas'):
            counts = pub.data.meas.get_counts()
        elif hasattr(pub.data, 'c'):
            counts = pub.data.c.get_counts()
        else:
            counts = pub.data[next(iter(pub.data._fields))].get_counts()

        total = sum(counts.values())
        print(f"\n  Counts ({total} shots):")
        for bs, cnt in sorted(counts.items(), key=lambda x: -x[1]):
            bar = chr(9608) * int(cnt / total * 50)
            print(f"    {bs}: {cnt:5d} ({cnt/total*100:5.1f}%) {bar}")

        target_hit_rate = counts.get('11', 0) / total
        print(f"\n  Target |11> hit rate: {target_hit_rate*100:.1f}%")
        print(f"  Classical random: 25%")
        RESULTS['ibm_grover'] = {'counts': counts, 'target_hit_rate': target_hit_rate, 'latency_s': dt}
    except Exception as e:
        dt = time.perf_counter() - t0
        print(f"  Timeout/Error after {dt:.1f}s: {e}")
        RESULTS['ibm_grover'] = {'job_id': job_grover.job_id(), 'status': str(job_grover.status()), 'error': str(e)[:120]}


## 9. VQE H2 — Ground State Energy (SF Simulation)


In [ ]:
print("=" * 70)
print("  VQE H2 — Ground State Energy")
print("=" * 70)

# H2 Hamiltonian (STO-3G)
H2_HAM = {'II': -0.4804, 'ZZ': 0.1712, 'XX': 0.0485, 'YY': -0.0485}

# Exact diagonalisation
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

H_mat = np.zeros((4, 4), dtype=complex)
for ps, coeff in H2_HAM.items():
    op = 1
    for ch in ps:
        m = {'I': I2, 'Z': Z, 'X': X, 'Y': Y}[ch]
        op = np.kron(op, m)
    H_mat += coeff * op
exact_energy = float(np.min(np.linalg.eigvalsh(H_mat)))
print(f"  Exact ground state energy (FCI): {exact_energy:.8f} Ha")

# VQE ansatz
def build_h2_ansatz(theta):
    qc = sf.Circuit(2)
    qc.h(0); qc.cx(0, 1)
    qc.ry(theta[0], 0); qc.ry(theta[1], 1)
    return qc

def energy_sf(theta):
    qc = build_h2_ansatz(theta)
    be = BackendRegistry.get_backend("statevector")
    sv = np.asarray(be.run(qc, shots=0).statevector, dtype=np.complex128)
    e = 0.0
    for ps, coeff in H2_HAM.items():
        probs = np.abs(sv) ** 2
        idx = np.arange(4)
        parity = np.zeros(4, dtype=int)
        for bit, ch in enumerate(ps):
            if ch == 'Z':
                parity ^= (idx >> (3 - bit)) & 1
            elif ch == 'X':
                parity ^= ((idx >> (3 - bit)) & 1) ^ 1
            elif ch == 'Y':
                parity ^= ((idx >> (3 - bit)) & 1) ^ ((idx >> (3 - bit - 2)) & 1)
        sign = np.where(parity == 0, 1.0, -1.0)
        e += coeff * float(np.real(np.sum(sign * probs)))
    return e

# Run VQE
from scipy.optimize import minimize
print("\n  Running VQE optimisation (Nelder-Mead, 300 iters)...")
t0 = time.time()
result = minimize(energy_sf, x0=[0.1, 0.1], method='Nelder-Mead',
                  options={'maxiter': 300, 'xatol': 1e-8, 'fatol': 1e-8})
dt_vqe = time.time() - t0
vqe_energy = float(result.fun)
energy_err = abs(vqe_energy - exact_energy)

print(f"  VQE energy:     {vqe_energy:.10f} Ha")
print(f"  Exact energy:    {exact_energy:.10f} Ha")
print(f"  Error:           {energy_err:.2e} Ha")
print(f"  Chemical accuracy (1.6 mHa): {'YES' if energy_err < 0.0016 else 'NO'}")
print(f"  Optimisation:    {dt_vqe:.2f}s, {result.nit} iters")
print(f"  Optimal params:  t0={result.x[0]:.6f}, t1={result.x[1]:.6f}")

RESULTS['vqe'] = {'energy': vqe_energy, 'exact': exact_energy, 'error': energy_err,
                  'time_s': dt_vqe, 'iters': result.nit, 'params': result.x.tolist()}


## 10. QEC Analysis — Noise-Aware Code Selection


In [ ]:
print("=" * 70)
print("  QEC — NOISE-AWARE CODE SELECTION")
print("=" * 70)

if not IBM_TOKEN:
    print("  Skipped — no IBM token")
else:
    ibmq_backend = service.backend(TARGET_BACKEND)
    props = ibmq_backend.properties()
    nq = min(ibmq_backend.num_qubits, 20)

    print(f"\n  Noise analysis for {TARGET_BACKEND} ({nq} qubits):")
    qubit_scores = []
    for i in range(nq):
        t1 = props.t1(i) if props.t1(i) else 0
        t2 = props.t2(i) if props.t2(i) else 0
        try:
            ro_err = props.readout_error(i)
        except:
            ro_err = 0.05
        try:
            cx_err = props.gate_error('cx', [i, (i + 1) % nq])
        except:
            cx_err = 0.01
        score = t2 / (1 + ro_err + cx_err * 10)
        qubit_scores.append({'qubit': i, 't1_us': t1 * 1e6, 't2_us': t2 * 1e6,
                              'ro_err': ro_err, 'cx_err': cx_err, 'score': score})

    qubit_scores.sort(key=lambda x: -x['score'])

    print(f"\n  Top-5 best qubits:")
    print(f"  {'Qubit':<8s} {'T1(us)':<12s} {'T2(us)':<12s} {'RO Err':<10s} {'CX Err':<10s} {'Score':<10s}")
    print("  " + "-" * 64)
    for qs in qubit_scores[:5]:
        print(f"  {qs['qubit']:<8d} {qs['t1_us']:<12.1f} {qs['t2_us']:<12.1f} {qs['ro_err']:<10.4f} {qs['cx_err']:<10.4f} {qs['score']:<10.1f}")

    avg_ro_err = np.mean([q['ro_err'] for q in qubit_scores])
    avg_cx_err = np.mean([q['cx_err'] for q in qubit_scores])
    avg_t2 = np.mean([q['t2_us'] for q in qubit_scores])

    print(f"\n  Average metrics:")
    print(f"    T2: {avg_t2:.1f} us")
    print(f"    Readout error: {avg_ro_err:.4f}")
    print(f"    CX gate error: {avg_cx_err:.4f}")

    physical_error = avg_cx_err
    if physical_error < 0.001:
        recommended_d, code = 3, "Surface-3"
    elif physical_error < 0.005:
        recommended_d, code = 5, "Surface-5"
    elif physical_error < 0.01:
        recommended_d, code = 7, "Surface-7"
    else:
        recommended_d, code = 9, "Surface-9"

    print(f"\n  Recommended QEC code: {code} (distance d={recommended_d})")
    print(f"  Physical qubits per logical: ~{2 * recommended_d ** 2}")

    RESULTS['qec'] = {
        'avg_t2_us': avg_t2, 'avg_ro_err': avg_ro_err, 'avg_cx_err': avg_cx_err,
        'recommended_code': code, 'recommended_distance': recommended_d,
        'top_qubits': [q['qubit'] for q in qubit_scores[:5]]
    }


## 11. Stabilizer Backend — Large-Scale Clifford (n=20..100)


In [ ]:
print("=" * 70)
print("  STABILIZER BACKEND — Aaronson-Gottesman Tableau")
print("=" * 70)

if "stabilizer" not in working:
    print("  Stabilizer backend not available")
else:
    stab = StabilizerBackend()
    stab_results = []

    print(f"\n  {'n':<8s} {'Layers':<8s} {'Time(ms)':<14s} {'<Z0Z1>':<16s} {'Status':<10s}")
    print("  " + "-" * 60)

    for n in [10, 20, 50, 80, 100, 127]:
        try:
            circuit = make_clifford_sf(n, layers=min(20, n // 2))
            obs = "ZZ" + "I" * (n - 2)

            gc.collect()
            proc = psutil.Process(os.getpid())
            rss0 = proc.memory_info().rss
            t0 = time.perf_counter()
            z_val = stab.expval(circuit, obs)
            dt = (time.perf_counter() - t0) * 1000
            mem = (proc.memory_info().rss - rss0) / 1024 / 1024

            print(f"  {n:<8d} {min(20, n//2):<8d} {dt:<14.2f} {z_val:<+16.8f} {'OK':<10s}")
            stab_results.append({'n': n, 'time_ms': dt, 'mem_mb': mem, 'z_exp': float(z_val)})
        except Exception as e:
            print(f"  {n:<8d} {'-':<8s} {'FAIL':<14s} {'':<16s} {str(e)[:30]}")

    RESULTS['stabilizer'] = stab_results


## 12. MPS Backend — High-Qubit Scaling (20->100q)


In [ ]:
print("=" * 70)
print("  MPS BACKEND — High-Qubit Scaling")
print("=" * 70)

if "mps" not in working:
    print("  MPS backend not available")
else:
    mps_results = []
    print(f"\n  {'n':<8s} {'Time(ms)':<14s} {'Mem(MB)':<12s} {'MaxBond':<10s}")
    print("  " + "-" * 48)

    for n in [20, 30, 50, 80, 100]:
        try:
            circuit = make_qaoa_sf(n)
            mps_be = MPSSimulatorBackend(options={"max_bond_dim": 64})

            gc.collect()
            proc = psutil.Process(os.getpid())
            rss0 = proc.memory_info().rss
            t0 = time.perf_counter()
            res = mps_be.run(circuit, shots=128)
            dt = (time.perf_counter() - t0) * 1000
            mem = (proc.memory_info().rss - rss0) / 1024 / 1024
            bond = getattr(res, 'max_bond_dim', '?')

            print(f"  {n:<8d} {dt:<14.2f} {mem:<+12.2f} {str(bond):<10s}")
            mps_results.append({'n': n, 'time_ms': dt, 'mem_mb': mem, 'bond': str(bond)})
        except Exception as e:
            print(f"  {n:<8d} {'FAIL':<14s} {'':<12s} {str(e)[:30]}")

    RESULTS['mps_scaling'] = mps_results


## 13. SUMMARY — All Results


In [ ]:
print("=" * 70)
print("  SHOWCASE SUMMARY")
print("=" * 70)

print(f"\n  Working SF backends: {len(working)} — {working}")

# Cross-framework
if 'cross_validation' in RESULTS:
    cv = RESULTS['cross_validation']
    n_pass = sum(1 for r in cv if r['pass'])
    print(f"\n  Cross-framework (SF vs Aer): {n_pass}/{len(cv)} passed")
    for r in cv:
        status = "PASS" if r['pass'] else "FAIL"
        print(f"    {status} {r['name']:12s}  fid={r['fidelity']:.10f}  D<Z0Z1>={r['diff']:.2e}")

# VQE
if 'vqe' in RESULTS:
    v = RESULTS['vqe']
    chem = "CHEMICAL ACCURACY" if v['error'] < 0.0016 else "NOT within chemical accuracy"
    print(f"\n  VQE H2: E={v['energy']:.10f} Ha  DE={v['error']:.2e}  {chem}")
    print(f"    Time: {v['time_s']:.1f}s, {v['iters']} iters")

# IBM QPU
if 'ibm_bell' in RESULTS:
    ib = RESULTS['ibm_bell']
    if 'fidelity' in ib:
        print(f"\n  IBM Bell:  fid={ib['fidelity']*100:.1f}%  latency={ib['latency_s']:.1f}s")
    else:
        print(f"\n  IBM Bell:  job={ib.get('job_id', '?')}  status={ib.get('status', '?')}")

if 'ibm_ghz3' in RESULTS:
    ig = RESULTS['ibm_ghz3']
    if 'fidelity' in ig:
        print(f"  IBM GHZ-3: fid={ig['fidelity']*100:.1f}%  latency={ig['latency_s']:.1f}s")
    else:
        print(f"  IBM GHZ-3: job={ig.get('job_id', '?')}  status={ig.get('status', '?')}")

if 'ibm_grover' in RESULTS:
    igr = RESULTS['ibm_grover']
    if 'target_hit_rate' in igr:
        print(f"  IBM Grover: target hit={igr['target_hit_rate']*100:.1f}%  latency={igr['latency_s']:.1f}s")
    else:
        print(f"  IBM Grover: job={igr.get('job_id', '?')}  status={igr.get('status', '?')}")

# Stabilizer
if 'stabilizer' in RESULTS:
    st = RESULTS['stabilizer']
    if st:
        max_n = max(r['n'] for r in st)
        print(f"\n  Stabilizer: up to n={max_n} qubits simulated")

# MPS
if 'mps_scaling' in RESULTS:
    ms = RESULTS['mps_scaling']
    if ms:
        max_n = max(r['n'] for r in ms)
        print(f"  MPS scaling: up to n={max_n} qubits")

# QEC
if 'qec' in RESULTS:
    q = RESULTS['qec']
    print(f"\n  QEC: {q['recommended_code']} (d={q['recommended_distance']})")
    print(f"    Avg T2: {q['avg_t2_us']:.1f} us  RO err: {q['avg_ro_err']:.4f}  CX err: {q['avg_cx_err']:.4f}")

# Save results
out_path = ROOT / 'notebooks' / 'sf_ibm_showcase_results.json'
def safe_serialize(obj):
    if isinstance(obj, dict):
        return {str(k): safe_serialize(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [safe_serialize(x) for x in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(out_path, 'w') as f:
    json.dump(safe_serialize(RESULTS), f, indent=2, default=str)
print(f"\n  Results saved to: {out_path}")

print("\n" + "=" * 70)
print("  SHOWCASE COMPLETE")
print("=" * 70)
